<div align="center">
  <a href="https://colab.research.google.com/github/PrunaAI/ai-efficiency-courses/blob/main/solutions/05-benchmark_llm_bits.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

---
**💡 Tip**: Click the button above to open this notebook in Google Colab for free GPU access!

## Installation

This notebook includes automatic setup cells that will install the project from git repository with UV.

**Note**: Run the setup cells below before starting the exercises.

In [ ]:
# Install project directly from git repository
!uv pip install git+https://github.com/PrunaAI/ai-efficiency-courses.git

## Utility cells

During the course, we'll leverage some course utilities to streamline our workflow. These utilities are located in the `course` package, which can simply be imported given that we installed the project from git repository above.
You can find the source code [here](https://github.com/PrunaAI/ai-efficiency-courses/tree/main/course).

These utilities will help us:
- Load and manage lists of model ids that we have verified to work.
- Generate informative plots for model analysis.
- Iterate efficiently over evaluation and model configuration options.

Let's first load our models. We will use `SMALL_MODEL_IDS`, which are sub 1B parameters which should be easy to download and load into memory. We recommend starting with these smaller models but feel free to experiment with other models until you reach your GPU memory limit!

In [ ]:
from course import SMALL_MODEL_IDS, MEDIUM_MODEL_IDS, LARGE_MODEL_IDS, ALL_MODEL_IDS

MODEL_IDS = SMALL_MODEL_IDS
# MODEL_IDS = MEDIUM_MODEL_IDS
# MODEL_IDS = LARGE_MODEL_IDS
# MODEL_IDS = ALL_MODEL_IDS

MODEL_IDS

We also recommend to set a custom cache directory for models. Loading models can take significant disk space. To avoid filling up your default disk, we recommend setting a custom cache directory for downloaded models. You can do this by running the following in a terminal or in a notebook cell:

In [ ]:
# Replace <path_to_cache> with your desired cache path
import os

CACHE_PATH = "<path_to_cache>"
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH

You can also clear the cache by running the following cell:

In [ ]:
from course.models import clear_cache

clear_cache(CACHE_PATH)

# 05: Benchmark LLM bit precision

Welcome to the next lecture of the LLM Efficiency course! ⚡

In this tutorial, we’ll explore how bit precision impacts the efficiency and performance of Large Language Models (LLMs). Bit precision determines how numerical values (like weights and activations) are represented inside the model. Moving from higher precision (e.g., FP32) to lower precision (e.g., INT8 or INT4) can dramatically reduce memory usage and speed up inference—but it often comes with accuracy trade-offs. The content from the chapter 4 [slides](https://github.com/PrunaAI/ai-efficiency-courses/blob/main/slides/04-quantize_language_models.pdf) will help you to explore this notebook.

By the end of this lecture, you will:
- Understand the role of bit precision in LLM computation.
- Learn how to benchmark models under different precision levels.
- Compare popular quantization configurations and see how bit precision affects accuracy, memory, and speed.

Let’s get started on finding the best bit precision for your LLM!

## 1. Imports

As we've already installed the project, we can import the necessary libraries. We will be using `transformers` for this tutorial as interface to the model and tokenizer. On top of that, we will be using `pruna` for quantization and evaluation. We recommend to checkout the [Pruna documentation](https://docs.pruna.ai/en/stable/index.html) for access to AI efficiency functions.

In [3]:
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer

from pruna import SmashConfig, smash
from pruna.evaluation.metrics import (
    EnergyConsumedMetric,
    InferenceMemoryMetric,
    LatencyMetric,
    TorchMetricWrapper,
)

Beyond external libraries, this course comes with the `course` local package which contains a lot of utils that you can use in the notebooks. We will be using `evaluate_model` to evaluate a single model with the set of metrics,and `create_comparison_plots` for visualizing the results.

In [4]:
from course import evaluate_model, create_comparison_plots

## 2. Benchmark LLMs bit precision

We recommend to check the [compression guide](https://docs.pruna.ai/en/stable/docs_pruna/user_manual/configure.html) and the[quantization overview](https://docs.pruna.ai/en/stable/compression.html#quantizers) in the pruna documentation for the implementation details.

### 2.1 Evaluate Different Bit Precision for LLM.int8()

In this section, you’ll benchmark LLM.int8()–style quantization across the precisions it supports (with a full-precision model as a baseline).

**Why is this important?**
LLM.int8() replaces FP16/FP32 linear layers. This often delivers large memory savings with minimal quality loss and near-FP16 speed, especially on consumer GPUs. Understanding its parameters lets you tune the trade-off without jumping to more aggressive (and riskier) schemes.

**Your tasks:**
- Create a `evaluate_config` function that takes the model and using the `SmashConfig` compresses the model. Then use `evaluate_model` to perform the evaluation using the inference memory, energy consumed, perplexity and latency metrics.
- Define LLM.int8() quantization configurations for all bits that are available.
- Quantize and evaluate the base model with all the quantization configurations.
- Compare the results in the plots.

**Key questions to answer:**
- Which bit precision is the best/worst for quality on your evaluation set?
- What is the impact on speed (throughput and latency) as precision decreases?
- What is the impact on memory (peak VRAM and on-disk size) across precisions?

In [ ]:
def evaluate_config(model_id, smash_config, metrics=None, dataset="WikiText"):
    """Evaluate multiple quantization configurations on a model.

    Args:
        model_id: The base model to quantize and evaluate
        smash_config: SmashConfig object containing quantization settings
        metrics: List of metrics to evaluate (default: None)
        dataset: Name of dataset to evaluate on (default: "WikiText")

    Returns:
        dict: Results dictionary mapping quantizer names to evaluation metrics
    """

    model_results = {}

    ### To Complete ###

In [ ]:
# Select the model to be quantized
model_id = MODEL_IDS[0]

smash_configs = []

### To Complete ###

In [ ]:
llm_int8_evaluation_results = {}

### To Complete ###

### 2.2 Evaluate Different Bit Precision for Quanto

In this section, you’ll benchmark Quanto quantization across all bit widths your build supports with a full-precision baseline for reference.

**Why is this important?**
When only weights are quantized and optimized kernels are available, inference latency remains comparable, and device memory usage is roughly reduced in proportion to the bitwidth ratio.

**Your tasks:**
- Define Quanto quantization configurations for all bits that are available.
- Quantize and evalute the base model with all the quantization configurations.
- Compare the results in the plots.

**Key questions to answer:**
- Which bit precision is the best/worst for quality on your evaluation set?
- What is the impact on speed (throughput and latency) as precision decreases?
- What is the impact on memory (peak VRAM and on-disk size) across precisions?

In [ ]:
# Create list of SmashConfig with different weight bits
smash_configs = []

### To Complete ###

In [ ]:
quanto_evaluation_results = {}

### To Complete ###

### 2.3 Evaluate Different Bit Precision for HQQ

In this section, you’ll benchmark HQQ (High-Quality Quantization) across all supported bit widths, using a full-precision model as a baseline.

**Why is this important?**
Half-Quadratic Quantization (HQQ) leverages fast, robust optimization techniques for on-the-fly quantization, eliminating the need for calibration data.. With the right configuration, HQQ can retain quality while delivering large memory savings.

**Your tasks:**
- Define HQQ quantization configurations for all bits that are available.
- Quantize and evalute the base model with all the quantization configurations.
- Compare the results in the plots.

**Key questions to answer:**
- Which bit precision is the best/worst for quality on your evaluation set?
- What is the impact on speed (throughput and latency) as precision decreases?
- What is the impact on memory (peak VRAM and on-disk size) across precisions?

In [ ]:
# Create list of SmashConfig with different quantization methods
smash_configs = []

### To Complete ###

In [ ]:
hqq_evaluation_results = {}

### To Complete ###

## Conclusion: What We've Learned About Bit Precision in Quantization

In this module, we explored how to benchmark the different quantization methods and how to select the best bit precision for your use case. Here’s a recap of the key concepts:

- **Quantization Trade-offs:**
    - LLM.int8() is a good starting point for quantization when you want to reduce the memory usage of the model with minimal quality loss. However, latency increases significantly for higher bit widths. Compared to the other methods the gains are limited.
    - Quanto provides more flexibility and can be used to achieve better quality with less memory usage.
    - HQQ enables large memory savings, but requires careful tuning, for instance, at 2 bits the perplexity is very high.
- **General Observations:**
    - Benchmark using several metrics to capture the trade-offs and identify the best model for your use case.
- There is no “one-size-fits-all”: the optimal bit precision depends on your hardware, model, and application tolerance for accuracy loss.

### Next Steps: Data in Quantization

Now that you understand how to measure the efficiency of LLMs, the next segment will focus on a new aspect of **quantization**. You’ll learn about how data can be used during the quantization process to improve the model.

👉 **Continue to the next notebook:**

[06-use_data_llm_quantization.ipynb on GitHub](https://github.com/PrunaAI/ai-efficiency-courses/blob/main/exercises/06-use_data_llm_quantization.ipynb)

## ⭐ Bonus Exercise: Benchmark Bit Precision with Different Quantization Methods

As a bonus, extend your analysis to explore how different quantization methods perform across different bit precisions.

**Your tasks:**

1. Explore different quantization configurations for the different quantization methods.
2. Compare the previous results with the new ones.

** Key questions to consider:**
- Which quantization method performs the best for your use case?